# 07 · ReAct & Plan-and-Execute Agents

Two classic agent designs: reason-then-act loops and plan-first execution.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/07_react_and_plan_execute_agents.ipynb)

**Covered in this notebook**

- ReAct Agent
- Plan and Execute

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

## ReAct Agent

Customer Support Bot
Delivery support bot

#### Scenario: A delivery support bot. It does not know the refund amount until it first checks how late the order is. That dependency is exactly why ReAct exists: step 2 cannot be decided before step 1 returns.

In [ ]:
pip install langchain-groq langgraph -q

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

my_api_key = userdata.get('groq_api')

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
# Fake company database. In real life this is an API call.
ORDERS = {
    "NST1042": {"item": "Laptop Stand", "price": 2400, "days_late": 6},
    "NST1077": {"item": "Wireless Mouse", "price": 900, "days_late": 0},
}

def track_order(order_id: str) -> str:
    """Look up an order and report how many days late it is"""
    print("📦 track_order called:", order_id)
    order = ORDERS.get(order_id)
    if not order:
        return "Order not found"
    return f"{order['item']}, price {order['price']}, {order['days_late']} days late"

def calculate_refund(price: float, days_late: int) -> float:
    """Calculate refund: 5 percent of price per day late, capped at 30 percent"""
    print("💰 calculate_refund called:", price, days_late)
    percent = min(days_late * 5, 30)
    return price * percent / 100

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

tools = [track_order, calculate_refund]
llm_with_tools = llm.bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# The ReAct prompt. The reasoning instruction is doing real work here.
react_prompt = (
    "You are a delivery support agent. Solve the request step by step. "
    "Before each action, briefly state what information you still need. "
    "Use the tools to get facts, never guess numbers. Stop once you can answer."
)

def reason_node(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def react_route(state: State) -> Literal["tools", END]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

In [ ]:
builder = StateGraph(State)
builder.add_node("reason", reason_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "reason")
builder.add_conditional_edges("reason", react_route)
builder.add_edge("tools", "reason")

react_agent = builder.compile()

In [ ]:
query = "Order NST1042 is late. Check it and tell me my refund amount."

result = react_agent.invoke({
    "messages": [("system", react_prompt), ("user", query)]
})

print(result["messages"][-1].content)

📦 track_order called: NST1042
Your order **NST1042** is 6 days late.  
The refund policy is 5 % of the order price per day late, capped at 30 %.  

- 5 % × 6 days = 30 % of the price  
- 30 % of ₹2,400 = **₹720**

**Refund amount: ₹720**.


## Plan and Execute
#### Scenario: Writing a launch brief has known stages. Nothing needs to be discovered mid-task, so planning everything upfront gives cleaner structure than ReAct's step-by-step drift.

In [ ]:
pip install langchain-groq langgraph -q

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

my_api_key = userdata.get('groq_api')

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0,
    api_key = my_api_key
)

In [ ]:
planner_prompt = (
    "You are a planner. Break the task into 4 concrete steps. "
    "Reply ONLY as a numbered list, one step per line, like '1. ...'. "
    "No introduction, no explanation."
)

def parse_plan(text: str) -> list:
    steps = []
    for line in text.splitlines():
        line = line.strip()
        if line and line[0].isdigit():
            steps.append(line.split(".", 1)[1].strip())
    return steps

In [ ]:
class PlanState(TypedDict):
    task: str
    plan: list
    step_index: int
    results: list

In [ ]:
def planner_node(state: PlanState):
    text = llm.invoke(planner_prompt + "\n\nTask: " + state["task"]).content
    plan = parse_plan(text)

    print("🗂️ PLAN:")
    for i, s in enumerate(plan, 1):
        print(f"   {i}. {s}")

    return {"plan": plan, "step_index": 0, "results": []}

def executor_node(state: PlanState):
    i = state["step_index"]
    step = state["plan"][i]

    # Pass earlier results forward so steps build on each other
    previous = "\n".join(state["results"])
    context = f"\n\nWork completed so far:\n{previous}" if previous else ""

    print(f"⚙️ executing step {i + 1}: {step}")

    out = llm.invoke(
        f"Overall task: {state['task']}{context}\n\n"
        f"Carry out ONLY this step and report the result briefly:\n{step}"
    ).content

    return {"results": state["results"] + [out], "step_index": i + 1}

def more_steps(state: PlanState):
    if state["step_index"] < len(state["plan"]):
        return "executor"
    return END

In [ ]:
builder = StateGraph(PlanState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")
builder.add_conditional_edges("executor", more_steps)

plan_execute_agent = builder.compile()

In [ ]:
task = "Create a launch brief for a new 8-week Agentic AI course for working data scientists."

result = plan_execute_agent.invoke({
    "task": task,
    "plan": [],
    "step_index": 0,
    "results": []
})

for i, r in enumerate(result["results"], 1):
    print(f"\n===== STEP {i} OUTPUT =====")
    print(r)

🗂️ PLAN:
   1. Identify the target audience (working data scientists) and define clear learning objectives for the 8‑week Agentic AI course.
   2. Draft a detailed curriculum outline, including weekly modules, key topics, hands‑on projects, and assessment methods.
   3. Create a launch marketing plan: craft messaging, select channels (email, LinkedIn, webinars), and schedule promotional activities.
   4. Assemble launch logistics: finalize course materials, set up enrollment platform, coordinate instructors, and plan a kickoff event.
⚙️ executing step 1: Identify the target audience (working data scientists) and define clear learning objectives for the 8‑week Agentic AI course.
⚙️ executing step 2: Draft a detailed curriculum outline, including weekly modules, key topics, hands‑on projects, and assessment methods.
⚙️ executing step 3: Create a launch marketing plan: craft messaging, select channels (email, LinkedIn, webinars), and schedule promotional activities.
⚙️ executing step 4: A